# Independent validation — Milan / Hanoi / HCMC

Every predicted map, and the CLMS and GHSL products, against the 450 EarthLabel
plots of each city: independent photo-interpreted reference data (nine 10 m
sub-cells per plot, impervious share of the resolvable sub-cells). The maps come
from the registry in `output_paths.py` (34 rasters: 14 for Milan, 10 each for
Hanoi and HCMC).

Three techniques:
**A**, continuous agreement on the raw predicted % (RMSE, MAE, bias, R², Pearson r);
**B**, a 2×2 confusion matrix at a 50 % cut-off (OA, kappa, PA, UA, F1);
**C**, a 10-level confusion matrix (exact, within one level, QWK).

Two weighting schemes, both reported:
- **balanced** — every plot counts once, so each of the six strata (75 plots)
  carries 1/6: performance across the whole imperviousness gradient;
- **area-weighted** — each plot carries W_h / 75, W_h the share of the city's
  valid area in its stratum: performance representative of the city.

Uncertainty: a stratified bootstrap (1,000 iterations; inside each stratum 75
plots drawn with replacement; the **same** plot IDs for every map, so comparisons
are paired), 95 % percentile intervals for each map and for the difference
between the maps of each scientifically relevant pair. Bias is **observed minus
predicted**: positive means the map under-predicts.

## 0 · Setup — paths and maps

In [ ]:
import json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib, matplotlib.pyplot as plt
import rasterio
from pyproj import Transformer

import output_paths as op
import validation_metrics as vm

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

REPO    = op.REPO
GT      = REPO / "data" / "earthlabel"          # the final, adjudicated plot CSVs
RESULTS = op.VALIDATION
CROSS   = RESULTS / "cross_city"
B_ITER, B_SEED = 1000, 42
SCHEMES = vm.SCHEMES                            # ('balanced', 'area_weighted')

CITIES = ("Milan", "Hanoi", "HCMC")
DATASETS = {}
for city in CITIES:
    rasters = {m["key"]: m["raster"] for m in op.maps(city)}
    rasters.update(op.products(city))           # CLMS (Milan) and GHSL, the products themselves
    DATASETS[f"{city}_2018"] = dict(city=city, csv=GT / f"{city.lower()}_imd_2018_results_cells.csv",
                                    rasters=rasters)
DS_ORDER = list(DATASETS)

CITY_DIR = {ds: op.validation_dir(DATASETS[ds]["city"]) for ds in DS_ORDER}
for ds in DS_ORDER:
    for sub in (*SCHEMES, "bootstrap"):
        (CITY_DIR[ds] / sub).mkdir(parents=True, exist_ok=True)
    for r in DATASETS[ds]["rasters"]:
        (CITY_DIR[ds] / "per_raster" / r).mkdir(parents=True, exist_ok=True)
for sub in SCHEMES:
    (CROSS / sub).mkdir(parents=True, exist_ok=True)

n_pairs = sum(len(d["rasters"]) for d in DATASETS.values())
print(f"{len(DATASETS)} datasets, {n_pairs} (dataset, raster) pairs")
missing = []
for ds, d in DATASETS.items():
    print(f"  {ds:11s} csv={'ok' if d['csv'].exists() else 'MISSING'}")
    for r, p in d["rasters"].items():
        ok = p.exists()
        missing += [] if ok else [p]
        print(f"      {r:30s} {'ok' if ok else 'MISSING'}  {p.relative_to(REPO).as_posix()}")
assert not missing, f"{len(missing)} raster(s) missing"
print("results ->", RESULTS)

## 1 · Stratum areas and area weights

The six strata C0–C5 are the classes of the stratum raster the plots were drawn
from (`data/earthlabel/add_points/IMD_<City>_2018_reclass.tif`). Their areas are
counted inside the combined valid mask (`output/sampling/<City>_combined_valid_mask.tif`),
where the plots were drawn. W_h is each stratum's share of that area. These are
areas of the mapped strata, not true areas.

In [ ]:
AREAS, W = {}, {}
for ds in DS_ORDER:
    city = DATASETS[ds]["city"]
    a = vm.stratum_areas(REPO / "data" / "earthlabel" / "add_points" / f"IMD_{city}_2018_reclass.tif",
                         op.SAMPLING / f"{city}_combined_valid_mask.tif")
    a.insert(0, "dataset", ds)
    a.to_csv(CITY_DIR[ds] / "stratum_areas.csv", index=False)
    AREAS[ds] = a
    W[ds] = dict(zip(a.stratum, a.W_h))
pd.concat(AREAS.values()).to_csv(CROSS / "stratum_areas.csv", index=False)
display(pd.concat(AREAS.values()).set_index(["dataset", "stratum"]))

## 2 · Load the reference and extract predictions

Each plot's reference value is the impervious share of its resolvable sub-cells
(`validation_metrics.reference_from_cells`); its stratum is `ref_label`. Each
raster is sampled at the plot location reprojected to its CRS (one pixel;
nodata and values outside 0–100 masked).

In [ ]:
def load_dataset(ds):
    cfg = DATASETS[ds]
    raw = pd.read_csv(cfg["csv"])
    ref = [vm.reference_from_cells(js) for js in raw["cells_json"]]
    df = pd.DataFrame({
        "PLOTID": raw["PLOTID"], "LON": raw["LON"].astype(float), "LAT": raw["LAT"].astype(float),
        "gt_density_class": raw["ref_label"],
        "gt_dom_class": raw["class_label"].astype(str)
                            .str.replace("^PerviousPervious", "Pervious", regex=True)
                            .str.replace("Havily", "Heavily").str.strip(),
        "gt_dom_pct": raw["cell_dominant_pct"].astype(float),
        "n_unresolved_subcells": [n for _, n in ref],
        "gt_imperv_pct": np.round([v for v, _ in ref], 2),
    })
    df["gt_bin"] = (df.gt_imperv_pct >= vm.BIN_THRESHOLD).astype(int)
    assert set(df.gt_density_class) == set(vm.STRATA), set(df.gt_density_class)
    df["w_area"] = vm.plot_weights(df.gt_density_class, W[ds])

    tr_cache = {}
    for rname, rpath in cfg["rasters"].items():
        with rasterio.open(rpath) as src:
            crs = str(src.crs)
            if crs not in tr_cache:
                t = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
                tr_cache[crs] = np.column_stack(t.transform(df.LON.values, df.LAT.values))
            xy = tr_cache[crs]
            b = src.bounds
            inside = ((xy[:, 0] > b.left) & (xy[:, 0] < b.right) &
                      (xy[:, 1] > b.bottom) & (xy[:, 1] < b.top)).sum()
            vals = np.array([v[0] for v in src.sample(xy)], float)
            if src.nodata is not None:
                vals[vals == src.nodata] = np.nan
            vals[~np.isfinite(vals)] = np.nan
            vals[(vals < 0) | (vals > 100)] = np.nan
            df[f"pred_{rname}"] = vals
        print(f"  {ds:11s} {rname:30s} {crs}  inside {inside}/{len(df)}  nan {int(np.isnan(vals).sum())}")
    return df

DATA = {ds: load_dataset(ds) for ds in DS_ORDER}
for ds in DS_ORDER:
    DATA[ds].to_csv(CITY_DIR[ds] / "points_extracted.csv", index=False)

summ = pd.DataFrame([{
    "dataset": ds, "n_plots": len(DATA[ds]),
    "plots_per_stratum": DATA[ds].gt_density_class.value_counts().to_dict(),
    "gt_imperv_%_mean": DATA[ds].gt_imperv_pct.mean(),
    "gt_imperv_%_mean_area_weighted": np.average(DATA[ds].gt_imperv_pct, weights=DATA[ds].w_area),
    "binary_impervious": int(DATA[ds].gt_bin.sum()), "rasters": ", ".join(DATASETS[ds]["rasters"]),
} for ds in DS_ORDER]).set_index("dataset")
summ.to_csv(CROSS / "dataset_summary.csv")
display(summ)

def all_pairs():
    for ds in DS_ORDER:
        for r in DATASETS[ds]["rasters"]:
            yield ds, r

def weights(ds, scheme):
    return None if scheme == "balanced" else DATA[ds]["w_area"].to_numpy()

## 3 · Technique A — continuous agreement (raw %, no threshold)

`RMSE_pp` / `MAE_pp` in percentage points; `bias_pp` is **observed minus
predicted** (positive: the map under-predicts); `R2` against the 1:1 line;
`pearson_r` the linear correlation. Both schemes.

In [ ]:
TA = {}
for scheme in SCHEMES:
    rows = [dict(dataset=ds, raster=r, **vm.continuous_metrics(DATA[ds].gt_imperv_pct, DATA[ds][f"pred_{r}"],
                                                              weights(ds, scheme)))
            for ds, r in all_pairs()]
    TA[scheme] = pd.DataFrame(rows).set_index(["dataset", "raster"])
    TA[scheme].to_csv(CROSS / scheme / "technique_A_continuous_metrics.csv")
    for ds in DS_ORDER:
        TA[scheme].loc[ds].to_csv(CITY_DIR[ds] / scheme / "technique_A_continuous_metrics.csv")
print("bias_pp = observed - predicted  (positive => the map under-predicts)")
pd.concat(TA, names=["scheme"]).round(3)

### 3.1 · Predicted vs reference, one panel per map (balanced metrics in the titles)

In [ ]:
def grid_axes(n, w=3.3, h=3.6, ncol=5):
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, min(n, ncol), figsize=(w * min(n, ncol), h * nrow),
                             constrained_layout=True, squeeze=False)
    for ax in axes.ravel()[n:]:
        ax.axis("off")
    return fig, axes.ravel()[:n]

def continuous_grid(ds, savepath):
    rs = list(DATASETS[ds]["rasters"])
    fig, axes = grid_axes(len(rs))
    for i, (ax, r) in enumerate(zip(axes, rs)):
        d = DATA[ds]
        g = d.gt_imperv_pct.values.astype(float)
        p = np.clip(d[f"pred_{r}"].values.astype(float), 0, 100)
        st = TA["balanced"].loc[(ds, r)]
        ax.scatter(g, p, s=6, alpha=.35, edgecolors="none")
        ax.plot([0, 100], [0, 100], "r-", lw=.8, alpha=.6)
        ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.set_aspect("equal")
        ax.set_xlabel("reference impervious %"); ax.set_ylabel("predicted %")
        ax.set_title(f"{r}\nRMSE {st.RMSE_pp:.1f}  MAE {st.MAE_pp:.1f}\n"
                     f"bias {st.bias_pp:+.1f}  R2 {st.R2:.3f}", fontsize=8.5)
    fig.savefig(savepath, dpi=140, bbox_inches="tight"); plt.show()

for ds in DS_ORDER:
    continuous_grid(ds, CITY_DIR[ds] / "balanced" / "technique_A_continuous_grid.png")

## 4 · Technique B — confusion matrix at a 50 % cut-off

Impervious if ≥ 50 %, for the reference and the map. Balanced matrices are plot
counts; area-weighted matrices are estimated proportions of the city's valid area.

In [ ]:
TB, CMS = {}, {}
for scheme in SCHEMES:
    rows = []
    for ds, r in all_pairs():
        d = DATA[ds]
        cm, mm = vm.binary_metrics(d.gt_imperv_pct, d[f"pred_{r}"], weights(ds, scheme))
        CMS[(scheme, ds, r)] = cm
        rows.append(dict(dataset=ds, raster=r, **mm))
        out = cm if scheme == "balanced" else cm / cm.sum()
        name = "confusion_matrix_50pct.csv" if scheme == "balanced" else "confusion_matrix_50pct_area_weighted.csv"
        pd.DataFrame(out, index=["ref_pervious", "ref_impervious"],
                     columns=["pred_pervious", "pred_impervious"]).to_csv(CITY_DIR[ds] / "per_raster" / r / name)
    TB[scheme] = pd.DataFrame(rows).set_index(["dataset", "raster"])
    TB[scheme].to_csv(CROSS / scheme / "technique_B_hard_confusion_metrics.csv")
    for ds in DS_ORDER:
        TB[scheme].loc[ds].to_csv(CITY_DIR[ds] / scheme / "technique_B_hard_confusion_metrics.csv")
print(f"threshold = {vm.BIN_THRESHOLD:g} %")
pd.concat(TB, names=["scheme"]).round(3)

### 4.1 · Confusion-matrix heatmaps (balanced, plot counts)

In [ ]:
def confusion_grid(ds, savepath):
    rs = list(DATASETS[ds]["rasters"])
    fig, axes = grid_axes(len(rs), w=3.3, h=3.4)
    lab = ["perv.", "imperv."]
    for ax, r in zip(axes, rs):
        cm = CMS[("balanced", ds, r)]; mm = TB["balanced"].loc[(ds, r)]
        ax.imshow(cm, cmap="Blues", aspect="auto")
        for a in range(2):
            for c in range(2):
                ax.text(c, a, int(cm[a, c]), ha="center", va="center", fontsize=12,
                        color="white" if cm[a, c] > cm.max() / 2 else "black")
        ax.set_xticks([0, 1], lab); ax.set_yticks([0, 1], lab)
        ax.set_xlabel("predicted"); ax.set_ylabel("reference")
        ax.set_title(f"{r}\nOA {mm.OA:.3f}  \u03ba {mm.kappa:.3f}", fontsize=9)
    fig.savefig(savepath, dpi=140, bbox_inches="tight"); plt.show()

for ds in DS_ORDER:
    confusion_grid(ds, CITY_DIR[ds] / "balanced" / "technique_B_confusion_grid.png")

## 5 · Technique C — 10-level confusion matrix (no threshold)

The reference level is the impervious sub-cell count (0/9 … 9/9); the predicted
level is the raster value snapped to the nearest of the same ten steps.
Exact agreement, agreement within one level (one sub-cell, 11.1 pp), QWK, and
MAE / RMSE in percentage points. Both schemes; full matrices per raster.

In [ ]:
TC, LEVEL_MATS = {}, {}
for scheme in SCHEMES:
    rows = []
    for ds, r in all_pairs():
        d = DATA[ds]
        C, per, st = vm.level_confusion(d.gt_imperv_pct, d[f"pred_{r}"], weights(ds, scheme))
        LEVEL_MATS[(scheme, ds, r)] = (C, per)
        rows.append(dict(dataset=ds, raster=r, **st))
        sfx = "" if scheme == "balanced" else "_area_weighted"
        (C if scheme == "balanced" else C / C.values.sum()).to_csv(
            CITY_DIR[ds] / "per_raster" / r / f"level_confusion_matrix{sfx}.csv")
        per.round(4).to_csv(CITY_DIR[ds] / "per_raster" / r / f"level_confusion_perclass{sfx}.csv")
    TC[scheme] = pd.DataFrame(rows).set_index(["dataset", "raster"])
    TC[scheme].to_csv(CROSS / scheme / "technique_C_level_confusion_metrics.csv")
    for ds in DS_ORDER:
        TC[scheme].loc[ds].to_csv(CITY_DIR[ds] / scheme / "technique_C_level_confusion_metrics.csv")
pd.concat(TC, names=["scheme"]).round(3)

### 5.1 · Level-confusion heatmaps (balanced, plot counts)

In [ ]:
def level_confusion_grid(ds, savepath):
    rs = list(DATASETS[ds]["rasters"])
    fig, axes = grid_axes(len(rs), w=3.6, h=3.8)
    for ax, r in zip(axes, rs):
        C, _ = LEVEL_MATS[("balanced", ds, r)]; st = TC["balanced"].loc[(ds, r)]
        Cv = C.values
        ax.imshow(Cv, cmap="Blues", aspect="auto")
        for a in range(10):
            for b in range(10):
                if Cv[a, b]:
                    ax.text(b, a, int(Cv[a, b]), ha="center", va="center", fontsize=6,
                            color="white" if Cv[a, b] > Cv.max() / 2 else "black")
        ax.plot([-.5, 9.5], [-.5, 9.5], "r-", lw=.8, alpha=.6)
        ax.set_xticks(range(10), [f"{j}" for j in range(10)], fontsize=7)
        ax.set_yticks(range(10), [f"{j}/9" for j in range(10)], fontsize=7)
        ax.set_xlabel("predicted level (\u00b7/9)"); ax.set_ylabel("reference level")
        ax.set_title(f"{r}\nexact {st.exact_match_OA:.2f}  \u00b11 {st.within_1_level_OA:.2f}  "
                     f"QWK {st.quad_weighted_kappa:.2f}", fontsize=8)
    fig.savefig(savepath, dpi=140, bbox_inches="tight"); plt.show()

for ds in DS_ORDER:
    level_confusion_grid(ds, CITY_DIR[ds] / "balanced" / "technique_C_level_confusion_grid.png")

## 6 · Ranking per city, both schemes

Rank 1 = best on `R2` (A), `kappa` (B) and `quad_weighted_kappa` (C), then the
mean of the three ranks. A ranking ignores uncertainty: read it with the
intervals of Section 7 — maps whose intervals overlap strongly perform similarly.

In [ ]:
RANK = {}
for scheme in SCHEMES:
    tabs = {}
    for ds in DS_ORDER:
        sc = pd.DataFrame({"R2_continuous": TA[scheme].loc[ds, "R2"],
                           "kappa_50pct":   TB[scheme].loc[ds, "kappa"],
                           "QWK_levelconf": TC[scheme].loc[ds, "quad_weighted_kappa"]})
        ranks = sc.rank(ascending=False, method="min")
        ranks.columns = [c + "_rank" for c in ranks.columns]
        rk = pd.concat([sc, ranks], axis=1)
        rk["MEAN_RANK"] = ranks.mean(axis=1)
        rk = rk.sort_values("MEAN_RANK")
        tabs[ds] = rk
        rk.to_csv(CITY_DIR[ds] / scheme / "raster_ranking.csv")
    RANK[scheme] = tabs
    pd.concat(tabs, names=["dataset", "raster"]).to_csv(CROSS / scheme / "raster_ranking_all_cities.csv")
for ds in DS_ORDER:
    print(ds); display(pd.concat({s: RANK[s][ds]["MEAN_RANK"] for s in SCHEMES}, axis=1).round(2))

## 7 · Stratified, paired bootstrap

For each city, `B_ITER` iterations: inside each stratum C0–C5, 75 plots drawn
**with replacement** from that stratum's 75; the six groups recombined (450
observations). The same resampled plot IDs are used for every map, so every
comparison is paired. In each iteration the nine metrics are computed for every
map, once with equal weights (balanced) and once with area weights W_h / 75.

The 95 % interval is the 2.5th–97.5th percentile of the 1,000 values. The point
estimate is the metric on the original 450 plots, not the bootstrap mean. An
estimate can fall outside its percentile interval for a skewed statistic; that
is reported as a warning, not an error.

In [ ]:
EST, BOOT, CI = {}, {}, {}
for ds in DS_ORDER:
    maps = list(DATASETS[ds]["rasters"])
    EST[ds] = vm.point_estimates(DATA[ds], maps, W[ds])
    BOOT[ds] = vm.bootstrap(DATA[ds], maps, W[ds], B=B_ITER, seed=B_SEED)
    for scheme in SCHEMES:
        BOOT[ds][scheme].to_csv(CITY_DIR[ds] / "bootstrap" / f"iterations_{scheme}.csv", index=False)
        ci = vm.confidence_intervals(EST[ds][scheme], BOOT[ds][scheme])
        ci.insert(0, "dataset", ds); ci.insert(1, "scheme", scheme)
        ci.to_csv(CITY_DIR[ds] / "bootstrap" / f"ci_{scheme}.csv", index=False)
        CI[(ds, scheme)] = ci
        out = ci[ci.estimate_outside_ci]
        for _, row in out.iterrows():
            print(f"  WARNING {ds} {scheme}: {row.map} {row.metric} estimate {row.estimate:.4f} "
                  f"outside [{row.ci_lo:.4f}, {row.ci_hi:.4f}]")
    print(f"{ds}: {B_ITER} iterations x {len(maps)} maps x 2 schemes")
for scheme in SCHEMES:
    pd.concat([CI[(ds, scheme)] for ds in DS_ORDER]).to_csv(CROSS / f"ci_{scheme}.csv", index=False)

def ci_table(ds, scheme, metrics=("R2", "kappa", "quad_weighted_kappa", "RMSE_pp")):
    c = CI[(ds, scheme)]
    c = c[c.metric.isin(metrics)].copy()
    c["value"] = c.apply(lambda r: f"{r.estimate:.3f} [{r.ci_lo:.3f}, {r.ci_hi:.3f}]", axis=1)
    return c.pivot(index="map", columns="metric", values="value").reindex(DATASETS[ds]["rasters"])
for ds in DS_ORDER:
    for scheme in SCHEMES:
        print(f"{ds} · {scheme}"); display(ci_table(ds, scheme))

## 8 · Paired differences between the competing maps

Δ = metric(A) − metric(B) on the original plots, with the 95 % interval of Δ
over the paired bootstrap. Bias is compared as |bias|. `better_map` reads the
interval with the metric's direction (higher is better for R², OA, kappa,
exact, ±1 and QWK; lower for RMSE, MAE and |bias|); an interval that includes
0 means the two maps are indistinguishable with this sample (`similar`).

There are 31 pairs × 9 metrics × 2 schemes: about 5 % of the intervals would
exclude 0 by chance alone, so the verdicts are read mainly on the primary
metrics — R², kappa and QWK (`primary`).

Map letters used in the pair groups: Milan **1–4** CLMS points / CLMS label,
**5–8** CLMS points / GHSL label, **9–12** GHSL points / GHSL label (emb,
median, stack, percentile); Hanoi and HCMC **a, b, c** local (emb, median,
stack), **d, e** direct transfer of Milan CLMS points / CLMS, **f, g** of GHSL
points / GHSL, **h, i** of CLMS points / GHSL (emb, median).

In [ ]:
PRED = ("emb", "median", "stack", "percentile")
PAIRS = {"Milan_2018": (
    [("predictor: stack vs percentile", "stack_CLMSpts_CLMS", "percentile_CLMSpts_CLMS"),
     ("predictor: stack vs AlphaEarth", "stack_CLMSpts_CLMS", "emb_CLMSpts_CLMS")]
    + [(f"label at the same points ({p}): CLMS vs GHSL", f"{p}_CLMSpts_CLMS", f"{p}_CLMSpts_GHSL") for p in PRED]
    + [(f"sampling ({p}): CLMS vs GHSL points, GHSL label", f"{p}_CLMSpts_GHSL", f"{p}_GHSLpts_GHSL") for p in PRED])}

def vietnam_pairs(city):
    first = ([("predictor: local AlphaEarth vs local stack", "emb_local", "stack_local"),
              ("predictor: local AlphaEarth vs local median", "emb_local", "median_local")] if city == "Hanoi" else
             [("predictor: local AlphaEarth vs local median", "emb_local", "median_local"),
              ("local AlphaEarth vs median transfer (CLMS pts/CLMS)", "emb_local", "median_transfer_CLMSpts_CLMS"),
              ("local AlphaEarth vs median transfer (GHSL pts/GHSL)", "emb_local", "median_transfer_GHSLpts_GHSL")])
    rest = []
    for p in ("emb", "median"):
        rest += [(f"local vs transfer, same label ({p}): local vs GHSL pts/GHSL", f"{p}_local", f"{p}_transfer_GHSLpts_GHSL"),
                 (f"local vs transfer ({p}): local vs CLMS pts/CLMS", f"{p}_local", f"{p}_transfer_CLMSpts_CLMS"),
                 (f"transfer sampling ({p}): GHSL pts vs CLMS pts, GHSL label", f"{p}_transfer_GHSLpts_GHSL", f"{p}_transfer_CLMSpts_GHSL"),
                 (f"transfer label at the same points ({p}): CLMS vs GHSL", f"{p}_transfer_CLMSpts_CLMS", f"{p}_transfer_CLMSpts_GHSL")]
    return first + rest

PAIRS["Hanoi_2018"] = vietnam_pairs("Hanoi")
PAIRS["HCMC_2018"] = vietnam_pairs("HCMC")
print({ds: len(p) for ds, p in PAIRS.items()}, "pairs;", sum(map(len, PAIRS.values())), "in all")
for ds, pairs in PAIRS.items():
    for _, a, b in pairs:
        assert a in DATASETS[ds]["rasters"] and b in DATASETS[ds]["rasters"], (ds, a, b)

PAIRED = {}
for scheme in SCHEMES:
    for ds in DS_ORDER:
        pdiff = vm.paired_differences(EST[ds][scheme], BOOT[ds][scheme], PAIRS[ds])
        pdiff.insert(0, "dataset", ds); pdiff.insert(1, "scheme", scheme)
        pdiff.to_csv(CITY_DIR[ds] / "bootstrap" / f"paired_{scheme}.csv", index=False)
        PAIRED[(ds, scheme)] = pdiff
    allp = pd.concat([PAIRED[(ds, scheme)] for ds in DS_ORDER])
    allp.to_csv(CROSS / f"paired_{scheme}.csv", index=False)
    print(f"{scheme}: {len(allp)} intervals, {int(allp.ci_excludes_0.sum())} exclude 0 "
          f"({int(allp[allp.primary].ci_excludes_0.sum())} of {int(allp.primary.sum())} on the primary metrics)")

def verdicts(ds, scheme):
    p = PAIRED[(ds, scheme)]; p = p[p.primary].copy()
    p["value"] = p.apply(lambda r: f"{r.delta:+.3f} [{r.ci_lo:+.3f}, {r.ci_hi:+.3f}] {r.better_map}", axis=1)
    return p.pivot_table(index=["group", "map_a", "map_b"], columns="metric", values="value", aggfunc="first", sort=False)
for ds in DS_ORDER:
    for scheme in SCHEMES:
        print(f"{ds} · {scheme}"); display(verdicts(ds, scheme))

## 9 · Files written

In [ ]:
for f in sorted(RESULTS.rglob("*")):
    if f.is_file():
        print("  ", f.relative_to(RESULTS).as_posix())

## Notes

* The reference is the final, adjudicated photo-interpretation in
  `data/earthlabel/<city>_imd_2018_results_cells.csv`; agreement between the two
  interpreters before adjudication is computed by `interpreter_agreement.py`.
* Only the point sample at each plot is used. Every plot falls inside every
  raster; a map with no value at a plot is dropped from that map's metrics.
* All rasters are 10 m; Milan is EPSG:32632, Hanoi/HCMC EPSG:32648.